# ⚙️ Deterministic Tool Execution & Agentic Routing

**My Objective:** I am engineering a secure, deterministic function-calling pipeline to bridge stochastic LLM text generation with rigid external APIs and mathematical computations. Because LLMs do not execute code natively, I am implementing a robust round-trip orchestration layer that handles tool request parsing, strict type validation, intent routing to prevent over-calling, and autonomous self-correction loops.

In [ ]:
import sys, json, re
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "prompting").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "prompting"))
sys.path.insert(0, str(ROOT / "orchestration"))

from dotenv import load_dotenv
load_dotenv(override=True)

import os
from openai import OpenAI
from reasoning_utils import DiskCache

client = OpenAI(api_key=os.environ["API_KEY"], base_url=os.environ["BASE_URL"])
MODEL = os.environ["MODEL"]
cache = DiskCache(Path.home() / "Desktop" / "FSDS" / "gen_ai" / "applied-llm-ops" / "tools_cache.json")

from tool_tasks import BANKASSIST_SYSTEM, GOLDEN, RATE_CARD, TOOLS, TOOLS_GUIDANCE, ROUTER_PROMPT, WAYS
from tool_utils import (answer_question, assistant_message, calculate_emi, call_model, check_tools_work,
                        compare_fixes, execute_tool, fixes_table, parse_args, question_grid, route, run_with_tools,
                        save_tools_report, score_question, show_trace, tool_message, tools_markdown, validate_args)

print(f"model:    {MODEL}")
print(f"endpoint: {os.environ['BASE_URL']}")
print(f"tools:    {[t['function']['name'] for t in TOOLS]}")
print(f"golden:   {len(GOLDEN)} questions, {sum(1 for g in GOLDEN if g['expect_tools'])} need a tool")

## 1. The Fallacy of Native Computation
LLMs predict tokens; they do not compute mathematics. To demonstrate why function calling is structurally necessary, I queried the model for an EMI calculation without providing any external tools. The model will confidently generate a plausible—but mathematically incorrect—number.

In [ ]:
question = "What is the EMI on a 25 lakh home loan at 8.5% for 20 years?"
out = call_model(client, MODEL, [{"role": "system", "content": BANKASSIST_SYSTEM},
                                 {"role": "user", "content": question}], tools=None, cache=cache)
print("model, no tools:", out["content"])
print()
exact = calculate_emi(2_500_000, 8.5, 240)
print(f"exact, by formula: Rs {exact:,.2f}  ->  Rs {round(exact):,} a month")
numbers = [float(n.replace(",", "")) for n in re.findall(r"\d[\d,]*\.?\d*", out["content"])]
print("numbers in the model's reply:", numbers)

## 2. The Execution Round-Trip
The model never executes functions itself; it outputs a structured JSON request. The orchestration code must:
1. Receive the `tool_calls` request.
2. Parse the arguments and execute the native Python code.
3. Append both the assistant's request and a new `tool` message (containing the result and matching `tool_call_id`) back into the conversation history.
4. Re-prompt the model to generate a natural language response using the injected factual data.

In [ ]:
ONE_TOOL = [TOOLS[0]]   
messages = [{"role": "system", "content": BANKASSIST_SYSTEM}, {"role": "user", "content": question}]

first = call_model(client, MODEL, messages, tools=ONE_TOOL, cache=cache)
print("finish_reason:", first["finish_reason"])
print("tool_calls:   ", first["tool_calls"])

if not first["tool_calls"]:
    print("No tool call this time - the model answered in prose:", first["content"][:200])
else:
    call = first["tool_calls"][0]
    args = parse_args(call["arguments"])            
    result = execute_tool(call["name"], args or {})  
    print("ran:", call["name"], args, "->", result)

    messages.append(assistant_message(first))       
    messages.append(tool_message(call["id"], result))   
    print("\n", json.dumps(messages[1:], indent=1, ensure_ascii=False))

    final = call_model(client, MODEL, messages, tools=ONE_TOOL, cache=cache)
    print("\nanswer:", final["content"])

## 3. Vulnerability Analysis: Over-Calling & Argument Hallucination
Without strict boundaries, models (especially smaller parameters) suffer from severe tool misuse. When asked a question irrelevant to available tools (e.g., branch timings), the model will often "over-call" the EMI calculator, hallucinating default arguments like `0` or injecting invalid string types into integer fields.

In [ ]:
timings = "What are your branch timings on Saturday?"
out = call_model(client, MODEL, [{"role": "system", "content": BANKASSIST_SYSTEM},
                                 {"role": "user", "content": timings}], tools=ONE_TOOL, cache=cache)
print("finish_reason:", out["finish_reason"])
print("tool_calls:   ", out["tool_calls"])
print("text:         ", out["content"][:200] or "(empty)")
for c in out["tool_calls"]:
    clean, problem = validate_args(c["name"], c["arguments"])
    print()
    print("validate_args says:", "OK" if problem is None else f"INVALID - {problem}")
    raw = parse_args(c["arguments"]) or {}
    print("run it unchecked:  ", execute_tool(c["name"], raw))

## 4. Multi-Step Orchestration
Enterprise queries often require sequential operations. I implemented a restricted loop that allows the model to chain multiple tool calls (e.g., fetch the rate, then calculate the EMI) but strictly capped at 3 steps to prevent infinite execution loops.

In [ ]:
two_step = "What would my EMI be on a 30 lakh home loan at your floating rate, over 15 years?"
res = run_with_tools(client, MODEL, [{"role": "system", "content": BANKASSIST_SYSTEM},
                                     {"role": "user", "content": two_step}], tools=TOOLS, cache=cache)
print(show_trace(res))
print()
print(f"model calls: {res['steps']}   tokens: {res['prompt_tokens'] + res['completion_tokens']:,}   "
      f"messages in the list now: {len(res['messages'])}")

## 5. Defense Mechanics: Prompting, Validation, and Intent Routing
To mitigate argument hallucination and over-calling, I engineered three independent fixes:
* **Fix A (Prompt Guidance):** Adding explicit behavioral constraints to the system prompt.
* **Fix B (Code-Level Validation):** Treating tool requests as untrusted external inputs. `validate_args` checks types and boundaries. If invalid, the application rejects execution and feeds the error string back to the model, forcing a self-correction loop.
* **Fix C (Intent Routing):** Implementing a cheap, preliminary triage call to classify the intent as `TOOL` or `NO_TOOL`. Only `TOOL` queries receive the function schemas, mechanically preventing over-calling on general FAQs.

In [ ]:
for name, raw in [("calculate_emi", '{"principal":"0","annual_rate":"0","tenure_months":"0"}'),
                  ("calculate_emi", '{"principal": 2500000, "annual_rate": 0.085, "tenure_months": 240}'),
                  ("get_rate", '{"product": "personal_loan", "rate_type": "floating"}')]:
    print(f"{name}({raw})\n   -> {validate_args(name, raw)[1]}")
print("=" * 90)
print(show_trace(answer_question(client, MODEL, timings, "validate", cache=cache)))

In [ ]:
print(ROUTER_PROMPT)
print()
for q in [question, two_step, timings, "Thanks, that's all I needed today!"]:
    needs, tokens = route(client, MODEL, q, cache=cache)
    print(f"{'TOOL   ' if needs else 'NO_TOOL'}  ({tokens:>5} tokens)  {q}")
print("=" * 90)
print(show_trace(answer_question(client, MODEL, timings, "router", cache=cache)))

## 6. Pipeline Evaluation (Golden Set Benchmarking)
I benchmarked the baseline implementation against the three fixes using a 10-question Golden Set. The evaluation matrix measures tool selection accuracy, argument validity, and final answer correctness.

In [ ]:
results = compare_fixes(client, MODEL, cache=cache)
print(fixes_table(results))
print("\n" + question_grid(results))

## 7. Serialization & CI/CD Checkpoint

In [ ]:
WORK_REPO = os.getenv("WORK_REPO") or r"C:\Users\HP\Desktop\FSDS\gen_ai\applied-llm-ops"   
report_text = tools_markdown(results, MODEL)
print(report_text)

SAVE_REPORT = True   
if SAVE_REPORT:
    path = save_tools_report(WORK_REPO, report_text)
    print(f"saved: {path}")
    
from prompt_utils import results_table
print(results_table(check_tools_work(WORK_REPO)))